# 2. Cleaning

Political outlets → text re-extracted from the HTML (trafilatura, precision) → English, 100–1,500 words, from 2025 →
no exact or near duplicates across outlets → at least 10 articles per outlet → one outlet per ownership group →
NewsGuard 2026 score and level (l1 < 40, l2 40–59, l3 60–74, l4 75–99, l5 = 100).

In [ ]:
N_JOBS = 32
NEAR_DUPLICATE = 0.5          # Jaccard of word 5-grams
CANDIDATES = 0.3              # MinHash looks for candidate pairs from this similarity (approximate), then >= NEAR_DUPLICATE
MIN_WORDS, MAX_WORDS = 100, 1500
MIN_ARTICLES = 10
FROM_DATE = "2025-01-01"

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import hashlib

from joblib import Parallel, delayed

import cleaning as CL

steps = []
def step(name, df):
    steps.append({"step": name, "articles": len(df), "outlets": df.domain.nunique()})
    print(f"{name}: {len(df):,} articles, {df.domain.nunique():,} outlets")

## 2.1 Political outlets and text extraction

In [ ]:
ng = CL.read_newsguard(C.NEWSGUARD)
outlets = CL.political_outlets(CL.read_status(C.SCRAPING), ng)
cache = C.OUT / "extracted.csv.gz"
if cache.exists():
    raw = pd.read_csv(cache)
else:
    rows = Parallel(n_jobs=N_JOBS)(delayed(CL.extract_outlet)(d, C.SCRAPING) for d in outlets)
    raw = pd.DataFrame([r for rr in rows for r in rr]); raw.to_csv(cache, index=False)
step("articles of political outlets", raw)
art = raw[raw.text.notna()].reset_index(drop=True)
step("with extracted text", art)

## 2.2 Language, length, date

In [ ]:
art["language"] = Parallel(n_jobs=N_JOBS, batch_size=200)(delayed(CL.language)(t) for t in art.text)
art = art[art.language == "en"]
step("English", art)
art["words"] = art.text.str.split().str.len()
art = art[art.words.between(MIN_WORDS, MAX_WORDS)]
step(f"{MIN_WORDS}-{MAX_WORDS:,} words", art)
art["date"] = pd.to_datetime(art.date, errors="coerce")
art = art[art.date >= FROM_DATE].reset_index(drop=True).rename_axis("article_id").reset_index()
step(f"published from {FROM_DATE[:4]}", art)

## 2.3 Duplicates and near duplicates (across all outlets, the oldest copy is kept)

In [ ]:
art["norm"] = art.text.map(CL.normalize)
art["fingerprint"] = art.norm.map(lambda s: hashlib.md5(s.encode()).hexdigest())
art = art.sort_values(["date", "article_id"]).drop_duplicates("fingerprint").sort_values("article_id")
step("no exact duplicates", art)
pairs = CL.similar_pairs(art.reset_index(drop=True), CANDIDATES, N_JOBS)
pairs = pairs[pairs.jaccard >= NEAR_DUPLICATE]
print(f"near-duplicate pairs: {len(pairs):,}, of which across outlets: {(pairs.outlet_a != pairs.outlet_b).sum():,}")
art = art[~art.article_id.isin(CL.copies_to_drop(pairs, art.set_index("article_id").date))]
step(f"no near duplicates (Jaccard >= {NEAR_DUPLICATE})", art)

## 2.4 Enough articles, one outlet per ownership group

In [ ]:
n = art.groupby("domain").size()
art = art[art.domain.isin(n.index[n >= MIN_ARTICLES])]
step(f"outlets with >= {MIN_ARTICLES} articles", art)
art["group"] = art.domain.map(CL.ownership_groups(sorted(art.domain.unique()), ng))
kept = (art.groupby("domain").agg(group=("group", "first"), n=("article_id", "size")).reset_index()
        .sort_values(["n", "domain"], ascending=[False, True]).groupby("group").head(1).domain)
art = art[art.domain.isin(kept)]
step("one outlet per ownership group", art)

## 2.5 Labels and dataset

In [ ]:
art["score"] = art.domain.map(ng.Score).astype(float)
art["label"] = C.level_from_score(art.score)
art["level"] = art.label.map(dict(enumerate(C.LEVELS)))
outlets = art.groupby("domain").agg(score=("score", "first"), label=("label", "first"), level=("level", "first"),
                                    articles=("article_id", "size"))
outlets["orientation"] = outlets.index.map(ng.Orientation).fillna("none")
art[["article_id", "domain", "url", "title", "date", "author", "text", "score", "label", "level"]] \
    .to_csv(C.OUT / "articles.csv.gz", index=False)
outlets.to_csv(C.OUT / "outlets.csv")
pd.DataFrame(steps).set_index("step")

In [ ]:
pd.DataFrame({"outlets": outlets.level.value_counts(), "articles": art.level.value_counts()}).reindex(C.LEVELS).T

## Check: same dataset as attempt 16 (same rules, same data)

In [ ]:
old = C.DATA / "tentativi" / "16_DataPreparation" / "articoli.csv.gz"
if old.exists():
    o = pd.read_csv(old)
    m = art.merge(o, on="article_id", suffixes=("", "_16"))
    print(f"articles {len(art):,} vs {len(o):,}; outlets {art.domain.nunique()} vs {o.domain.nunique()}; same ids: {set(art.article_id) == set(o.article_id)}")
    print({c: f"{(m[c].astype(str) == m[c16].astype(str)).mean():.2%}" for c, c16 in
           [("domain", "domain_16"), ("text", "text_16"), ("score", "score_16"), ("label", "label_16")]})